In [1]:
import os
import pandas as pd
import numpy as np

if os.path.exists("data"):
    DATA_DIR = "data"
elif os.path.exists("../data"):
    DATA_DIR = "../data"
else:
    raise FileNotFoundError("❌ Папка data не найдена!")

print(f"✅ Папка данных: {DATA_DIR}")

✅ Папка данных: ../data


In [2]:
df_features = pd.read_csv(f"{DATA_DIR}/features.csv")
print(f"Загружено {len(df_features)} пользователей")
df_features

Загружено 10 пользователей


,userId,income,expense,debt_ratio,balance,active_cards,days_on_platform,closed_credits,has_overdue
0,9,35000.000000,17066.666667,0.487619,178550.000000,2,225,0,0
1,10,20000.000000,2766.666667,0.138333,225400.000000,1,209,0,0
2,1,21666.666667,8832.817011,0.407668,145140.548968,5,375,4,0
3,2,21666.666667,6333.333333,0.292308,175300.000000,2,345,0,0
4,3,35000.000000,12333.333333,0.352381,146650.000000,3,315,1,0
5,4,15000.000000,2150.000000,0.143333,210250.000000,1,295,0,0
6,5,40000.000000,16000.000000,0.400000,183800.000000,1,285,0,0
7,6,18333.333333,2300.000000,0.125455,177000.000000,1,270,0,0
8,7,30000.000000,5066.666667,0.168889,229742.100000,2,255,1,0
9,8,16666.666667,3616.666667,0.217000,135050.000000,1,240,0,0


In [3]:
df_features['default'] = df_features['has_overdue'].apply(lambda x: 1 if x == 1 else 0)

print("📊 Распределение целевой переменной:")
print(df_features['default'].value_counts())
print(f"\nДефолтов: {df_features['default'].sum()} из {len(df_features)}")

📊 Распределение целевой переменной:
default
0    10
Name: count, dtype: int64

Дефолтов: 0 из 10


In [4]:
df_features.to_csv(f"{DATA_DIR}/features_with_target.csv", index=False)
print(f"✅ Сохранено в {DATA_DIR}/features_with_target.csv")

✅ Сохранено в ../data/features_with_target.csv


In [5]:
np.random.seed(42)
N = 1000

synthetic = pd.DataFrame({
    'income': np.random.normal(22000, 8000, N).clip(5000, 50000),
    'expense': np.random.normal(8000, 4000, N).clip(1000, 25000),
    'balance': np.random.normal(180000, 50000, N).clip(0, 400000),
    'active_cards': np.random.choice([1, 2, 3, 4, 5], N, p=[0.4, 0.3, 0.15, 0.1, 0.05]),
    'days_on_platform': np.random.randint(200, 400, N),
    'closed_credits': np.random.choice([0, 1, 2, 3, 4], N, p=[0.5, 0.25, 0.15, 0.05, 0.05]),
    'has_overdue': np.random.choice([0, 1], N, p=[0.9, 0.1]),
})

synthetic['debt_ratio'] = (synthetic['expense'] / synthetic['income']).clip(0, 1.0)

synthetic['default_prob'] = (
    0.05
    + 0.35 * (synthetic['debt_ratio'] > 0.5).astype(int)
    + 0.25 * synthetic['has_overdue']
    + 0.10 * (synthetic['balance'] < 100000).astype(int)
    + 0.05 * (synthetic['closed_credits'] == 0).astype(int)
)
synthetic['default'] = (np.random.random(N) < synthetic['default_prob']).astype(int)
synthetic = synthetic.drop('default_prob', axis=1)

print(f"✅ Синтетический датасет: {synthetic.shape}")
print(f"\n📊 Распределение default:")
print(synthetic['default'].value_counts())
print(f"\nДоля дефолтов: {synthetic['default'].mean():.1%}")
print(f"\nСредний доход: {synthetic['income'].mean():.0f}")
print(f"Средний расход: {synthetic['expense'].mean():.0f}")
synthetic.head(10)

✅ Синтетический датасет: (1000, 9)

📊 Распределение default:
default
0    767
1    233
Name: count, dtype: int64

Доля дефолтов: 23.3%

Средний доход: 22181
Средний расход: 8345


,income,expense,balance,active_cards,days_on_platform,closed_credits,has_overdue,debt_ratio,default
0,25973.713224,13597.421746,146241.086251,3,234,0,0,0.523507,1
1,20893.885591,11698.534732,172774.066464,5,313,1,0,0.559902,0
2,27181.508305,8238.521480,140379.003950,1,324,1,0,0.303093,0
3,34184.238851,5412.252889,164601.923518,5,328,1,0,0.158326,0
4,20126.773002,10792.893254,85319.266652,2,251,0,1,0.536246,1
5,20126.904344,9573.941542,190664.685369,1,284,1,0,0.475679,0
6,34633.702524,11580.772880,180060.273768,1,397,0,0,0.334379,0
7,28139.477833,10540.687207,139145.568447,1,364,1,0,0.374587,1
8,18244.204913,12198.210861,212962.283422,2,264,1,0,0.668607,0
9,26340.480349,5859.059154,226878.506881,2,378,0,0,0.222436,0


In [6]:
synthetic.to_csv(f"{DATA_DIR}/synthetic_data.csv", index=False)
print(f"✅ Сохранено в {DATA_DIR}/synthetic_data.csv")

✅ Сохранено в ../data/synthetic_data.csv
